# 01 — ARC-AGI Baseline (Qwen2.5-7B-Instruct, zero-shot)

**Goal:** establish a no-training baseline so we know what training buys us.

**Pipeline overview:**
1. **This notebook (baseline)** — zero-shot Qwen on ARC-AGI eval. ~10–20% expected.
2. `02_train_h100.ipynb` — QLoRA SFT on Qwen2.5-14B with BARC + RE-ARC + augmentations.
3. `03_distill_t4x2.ipynb` — distill the 14B teacher into Qwen2.5-3B for cheap inference.
4. `04_eval_ttt.ipynb` — test-time training (per-task LoRA) at inference.

**Hardware:** Colab T4 or Kaggle T4 is enough (4-bit base + greedy decoding).


In [ ]:
# Hardware check first
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


In [ ]:
# Install dependencies (Colab/Kaggle)
!pip -q install -U transformers==4.46.2 accelerate==1.1.1 bitsandbytes==0.44.1 datasets==3.1.0 peft==0.13.2


In [ ]:
# Download ARC-AGI-1 dataset (training + public eval)
import os, json, urllib.request, zipfile, io
from pathlib import Path

DATA_DIR = Path("/content/arc_data") if os.path.exists("/content") else Path("./arc_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "training").exists():
    url = "https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip"
    print("Downloading ARC-AGI repo...")
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
        z.extractall(DATA_DIR)
    # Move data/training and data/evaluation up
    src = DATA_DIR / "ARC-AGI-master" / "data"
    (DATA_DIR / "training").mkdir(exist_ok=True)
    (DATA_DIR / "evaluation").mkdir(exist_ok=True)
    for kind in ["training", "evaluation"]:
        for f in (src / kind).glob("*.json"):
            (DATA_DIR / kind / f.name).write_bytes(f.read_bytes())

train_files = sorted((DATA_DIR / "training").glob("*.json"))
eval_files  = sorted((DATA_DIR / "evaluation").glob("*.json"))
print(f"Train tasks: {len(train_files)}, Eval tasks: {len(eval_files)}")


In [ ]:
# Load model in 4-bit (fits T4 16GB easily)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
print("Model loaded.")


## Grid <-> string serialization

Each ARC cell is an integer 0–9 (color). We render grids as space-separated rows:

```
0 0 1
0 1 0
1 0 0
```

This is the standard format used by most public ARC fine-tunes (BARC, MindsAI, Greenblatt).


In [ ]:
def grid_to_str(g):
    return "\n".join(" ".join(str(c) for c in row) for row in g)

def str_to_grid(s):
    rows = []
    for line in s.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            row = [int(c) for c in line.split()]
            rows.append(row)
        except ValueError:
            return None
    if not rows:
        return None
    width = len(rows[0])
    if any(len(r) != width for r in rows):
        return None
    return rows

def format_task(task, include_test_output=False):
    parts = ["You are solving an ARC puzzle. Each grid is a 2D array of digits 0-9 (colors).",
             "Find the transformation rule from the input/output examples, then apply it to the test input.",
             ""]
    for i, ex in enumerate(task["train"], 1):
        parts.append(f"Example {i} Input:")
        parts.append(grid_to_str(ex["input"]))
        parts.append(f"Example {i} Output:")
        parts.append(grid_to_str(ex["output"]))
        parts.append("")
    test = task["test"][0]
    parts.append("Test Input:")
    parts.append(grid_to_str(test["input"]))
    parts.append("")
    parts.append("Test Output:")
    if include_test_output:
        parts.append(grid_to_str(test["output"]))
    return "\n".join(parts)


In [ ]:
# Inference helper — greedy, output-only parsing
@torch.inference_mode()
def predict(task, max_new_tokens=2048):
    prompt = format_task(task, include_test_output=False)
    messages = [
        {"role": "system", "content": "You are an expert ARC puzzle solver. Output only the test output grid as digits separated by spaces, one row per line. No explanations."},
        {"role": "user",   "content": prompt},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tok.eos_token_id,
    )
    completion = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return str_to_grid(completion), completion


In [ ]:
# Run baseline on first N eval tasks (full 400 takes a while on T4)
N = 50  # bump to 400 for full eval
results = []
correct = 0
for i, fp in enumerate(eval_files[:N]):
    task = json.loads(fp.read_text())
    pred, raw = predict(task)
    truth = task["test"][0]["output"]
    is_correct = pred == truth
    correct += int(is_correct)
    results.append({"task": fp.stem, "correct": is_correct, "pred": pred})
    if (i + 1) % 10 == 0:
        print(f"[{i+1}/{N}] accuracy so far: {correct}/{i+1} = {correct/(i+1):.1%}")

print(f"\nFINAL BASELINE: {correct}/{N} = {correct/N:.1%}")


In [ ]:
# Save baseline for comparison after training
import json
with open(DATA_DIR / "baseline_results.json", "w") as f:
    json.dump({"n": N, "correct": correct, "accuracy": correct / N, "details": results}, f, indent=2)
print("Saved baseline_results.json")


## What to expect

- Qwen2.5-7B zero-shot on ARC-AGI eval: typically **5–15%**.
- Random / naive baselines: ~0%.
- After training (notebook 02): target **30–55%**.
- With test-time training (notebook 04): target **50–60%+**.

The state of the art (o3, Greenblatt's TTT) is **80%+**, but those use orders of magnitude more compute. Our pipeline targets a strong open-source result you can actually reproduce.
